# Cross-family L23 check — is the L17 anomaly real, or a bad donor layer?

Donor `meta-llama/Llama-3.1-8B` -> recipient `google/gemma-2-2b`, recipient layer fixed at
`L_R = 20`. The completed run used **`L_D = 17`** and produced two results that contradict every
other model pair. This notebook re-runs the identical pair at **`L_D = 23`** and reports the same
quantities so they can be read side by side.

## The two anomalies at L_D = 17

**Anomaly 1 — the task map landed ON the recipient's natural manifold.**
`recon_cos_taskmap = 0.824 [0.816, 0.832]`, barely below the reconstruction map's
`0.906 [0.900, 0.912]`. In every other pair the task map's reconstruction cosine sits at
**0.20-0.27** (Gemma 0.197, Qwen 0.253, IT 0.266) while the recon map sits at 0.94-0.98 — i.e. the
task map leaves the manifold and the recon map does not. Yet conferral here was *high*
(leading-digit `0.843 [0.791, 0.884]`). That contradicts the paper's claim that *conferral coincides
with leaving the manifold*.

**Anomaly 2 — the stitched vector reads the answer WORSE than the recipient's own state.**
Transcription probe on the unsolvable bin (n=236): stitched **0.254** < recipient native **0.449**,
with donor reference 0.314 and recon map 0.297. In every other pair the stitched vector reads the
answer far *better* than the recipient's native state. Here it reads it worse — yet conferral is
high.

## Hypothesis: L17 is simply the wrong donor layer

The completed run's own donor-layer sweep says so:

| donor layer | donor leading-digit probe | task conferral (full answer) |
|---|---|---|
| **L17** | 0.275 [0.210, 0.352] | 0.195 [0.149, 0.250] |
| **L23** | **0.691 [0.613, 0.760]** | **0.263 [0.211, 0.322]** |
| L28 | 0.799 [0.727, 0.855] | 0.174 [0.131, 0.227] |

At L17 the donor state barely carries the answer linearly (probe 0.275, close to floor). If the
donor state has almost no linearly-readable answer signal, the CE objective has little gradient
pulling the map off its ridge-reconstruction warm start — so the trained map stays near that
initialization, which is exactly what a 0.824 cosine to the recipient manifold looks like. It could
still extract *something* a linear probe cannot see, which would explain high conferral alongside a
low stitched-vector probe. **L23 is where the donor's answer is most linearly available and where
full-answer conferral peaks**, so it is the fair test.

**Prediction if the hypothesis is right:** at L23 `recon_cos_taskmap` falls toward the 0.20-0.27
band seen in every other pair, and the stitched-vector probe rises above the recipient's native
0.449 (toward the donor's L23 probe of ~0.691). **If both anomalies survive at L23, they are real
properties of this cross-family pair, not an artifact of the donor layer.**

## What this notebook runs (and what it drops)

Kept: install, imports/config, HF login, stats helpers, shared helpers, the dual-tokenizer cell X0,
model loading, the arithmetic-bin setup, task-map training (5 seeds), the core channel eval
(recon / task / shuffle / self-graft), and the transcription probe — then a new comparison cell.

Dropped as unnecessary for this question: the donor-layer sweep (X2), free-vector ceiling (X5),
MLP map (X6), INLP erasure (X8), SAE (X9/X10), symbolic binding (X11).

Note: the donor-solved and recipient-unsolvable bins do **not** depend on `L_D` (the filters run
generation with no graft), so the bins should come out identical to the L17 run — n_eval_donor_solved
= 297, n_unsolvable = 236. That makes every number below directly comparable, same problems and all.

## Cross-family discipline (unchanged, do not weaken it)

Donor and recipient **do not share a tokenizer**. Every donor-side call uses `tokenizer_d` and
`ids_d`; every recipient-side call uses `tokenizer_r` / `tokenizer` and `ids_r`. Scoring is on
**decoded answer text only** — no token id is ever compared across families.

## Resources

**VRAM (48 GB card):** Llama-3.1-8B in bf16 ~16 GB + Gemma-2-2b in bf16 ~5 GB, plus generation KV
caches and the map's activations. Peak ~25-30 GB — comfortable headroom. The donor is **bf16, not
4-bit**: under 4-bit this transformers/bnb build runs unquantized layers in fp16 and Llama
activations overflow it (NaN logits). The NaN health-check assert in CELL 6 catches that.

**Expected runtime:** ~60-90 min end to end on a 48 GB card (~35-45 min of that is the 5-seed map
training: 5 seeds x 6 epochs x 3000 examples through the 2B recipient). Set `SMOKE_TEST = True` in
CELL 2 for a ~10 min integration check first if you want.

Run top-to-bottom. **ONE kernel restart after CELL 1.**

In [1]:
# === CELL 1: install (run once, then RESTART KERNEL) ===
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer
!pip uninstall -y torchvision torchaudio
# torchvision removed before any import (version mismatch crashes transformers). RESTART after this.
# >>> RESTART THE KERNEL NOW, then run every cell below in order. <<<
# --- Blackwell/sm_120 pods ONLY (verify cell prints cap (12,0)): uncomment, run, restart again ---
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124


In [1]:
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__, "(want 4.46.3)")

torch: 2.4.1+cu124 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)


In [2]:
# === CELL 2: imports, set_submodule shim, global config (CROSS-FAMILY donor-arm switch) ===
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)

# ---------------- CROSS-FAMILY CONFIG ----------------
# RECIPIENT is fixed; DONOR varies by family. Flip DONOR_ARM and re-run the notebook end to end.
MODEL_R = "google/gemma-2-2b"     # recipient (fixed)
L_R     = 20                      # recipient graft layer (validated in the within-family work)

DONOR_ARM = "llama"               # "qwen" (ARM A)  or  "llama" (ARM B)  <<< SET: LLAMA ARM
if DONOR_ARM == "qwen":
    MODEL_D = "Qwen/Qwen2.5-7B"          # 28 layers, d_model 3584
    L_D = 23                             # donor graft-layer start guess
    DONOR_SWEEP_LAYERS = [13, 18, 23, 25]
elif DONOR_ARM == "llama":
    MODEL_D = "meta-llama/Llama-3.1-8B"  # 32 layers, d_model 4096
    L_D = 17                             # donor graft-layer start guess
    # v1 found Llama's answer crystallizes DEEP, so the sweep must reach 23/28.
    DONOR_SWEEP_LAYERS = [17, 23, 28]
else:
    raise ValueError("DONOR_ARM must be 'qwen' or 'llama'")

PATCH_POS = -1                    # graft site = last prompt position (right-aligned by left padding)
RIDGE_LAMBDA = 1e3

# ---------------- experiment switches ----------------
RUN_LAYER_SWEEP = True    # EXP 1: donor-layer derivation (map sweep + per-layer donor probe)
RUN_CORE        = True    # EXP 2: recon / task(5 seeds) / SHUFFLE / self-graft
RUN_CEIL        = True    # EXP 3: per-example free-vector ceiling (capacity control)
RUN_MLP         = True    # EXP 4: nonlinear (MLP) map (capacity control)
RUN_TRANSCRIBE  = True    # EXP 5: leading-digit probe on donor state and on the stitched vector
RUN_ABLATE      = True    # EXP 6: INLP answer-subspace erasure + matched-rank random control
RUN_SAE         = True    # EXP 7: recipient-side GemmaScope feature delta at L_R
RUN_SYMBIND     = True    # EXP 8: symbolic binding chains, same channel battery

# Smoke test lever (True for a ~10 min integration check, False for the real run)
SMOKE_TEST = False

if SMOKE_TEST:
    N_ARITH_TRAIN, N_ARITH_EVAL = 200, 150
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
    CEIL_N, CEIL_STEPS = 24, 30
    G_INLP_ROUNDS = 8
    SYMBIND_DEPTHS = [1, 3]
    N_SYMBIND_TRAIN_PER_DEPTH, N_SYMBIND_EVAL_PER_DEPTH = 60, 40
else:
    N_ARITH_TRAIN, N_ARITH_EVAL = 3000, 2000     # -> a few hundred to ~1k in the unsolvable bin
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6 # 5 seeds for the task map
    BOOT_B = 10000
    CEIL_N, CEIL_STEPS = 200, 300                # free-vector ceiling is capped for runtime
    G_INLP_ROUNDS = 60
    SYMBIND_DEPTHS = [1, 3, 5]
    N_SYMBIND_TRAIN_PER_DEPTH, N_SYMBIND_EVAL_PER_DEPTH = 500, 350

SYMBIND_SEEDS = [0]           # channel battery on symbind uses one seed (recon/task/shuffle contrast)
ARITH_BATCH, MAX_NEW_ARITH = 16, 8
MAX_NEW_SYMBIND = 4
RESULTS = {}   # everything defensible gets concentrated here and printed/saved at the end
print("DONOR_ARM =", DONOR_ARM, "| donor =", MODEL_D, "L_D =", L_D,
      "| recipient =", MODEL_R, "L_R =", L_R, "| SMOKE_TEST =", SMOKE_TEST)

# ============================================================================
# === L23 CHECK OVERRIDE — the only deviation from the completed L17 run ===
# ============================================================================
# The donor arm above is already "llama" (donor = meta-llama/Llama-3.1-8B, L_D = 17).
# This notebook exists to re-run that SAME pair one layer-block deeper, where the donor's
# answer is actually linearly available (sweep: L23 probe 0.691 vs L17 probe 0.275).
# Recipient layer is UNCHANGED at L_R = 20 so the comparison is clean.
assert DONOR_ARM == "llama", "this notebook is the Llama-arm L23 check"
L_D = 23                                   # <<< THE OVERRIDE (was 17)
DONOR_SWEEP_LAYERS = [L_D]                 # sweep cell is dropped; keep the list consistent
QUANTIZE_DONOR = False                     # donor in bf16, NOT 4-bit (fp16 overflow -> NaN)

# Only the cells this notebook actually contains are switched on.
RUN_LAYER_SWEEP = False   # X2  dropped
RUN_CORE        = True    # X4  KEPT — recon / task(5 seeds) / shuffle / self-graft
RUN_CEIL        = False   # X5  dropped
RUN_MLP         = False   # X6  dropped
RUN_TRANSCRIBE  = True    # X7  KEPT — leading-digit probe on native / recon / stitched / donor
RUN_ABLATE      = False   # X8  dropped
RUN_SAE         = False   # X10 dropped
RUN_SYMBIND     = False   # X11 dropped

# ---- the completed L_D=17 run, verbatim from crossfamily_results_Llama-3.1-8B.json ----
# Hardcoded here so the comparison cell at the end can print old vs new side by side.
L17_REF = {
    "n_unsolv": 236, "n_eval_donor_solved": 297,
    "recon_cos_reconmap":   "0.906 [0.900, 0.912]",
    "recon_cos_taskmap":    "0.824 [0.816, 0.832]",   # ANOMALY 1
    "probe_native_L20":     "0.449 [0.362, 0.539]",
    "probe_recon_map":      "0.297 [0.222, 0.384]",
    "probe_stitched":       "0.254 [0.184, 0.340]",   # ANOMALY 2 (below native)
    "probe_donor_LD":       "0.314 [0.237, 0.402]",
    "donor_probe_at_LD":    "0.275 [0.210, 0.352]",
    "native_unsolv_lead":   "0.453 [0.391, 0.517]",
    "selfgraft_full":       "0.000 [0.000, 0.016]",
    "selfgraft_lead":       "0.449 [0.387, 0.513]",
    "recon_full":           "0.059 [0.036, 0.097]",
    "recon_lead":           "0.381 [0.322, 0.445]",
    "task_seed0_full":      "0.195 [0.149, 0.250]",
    "task_seed0_lead":      "0.843 [0.791, 0.884]",
    "task_full_acrossseed": "0.190 [0.157, 0.223]",
    "task_lead_acrossseed": "0.826 [0.811, 0.842]",
    "shuffle_full":         "0.008 [0.002, 0.030]",
    "shuffle_lead":         "0.140 [0.101, 0.190]",
}
# The L17 run's own donor-layer sweep (seed-0 maps, recipient layer fixed at 20).
L17_SWEEP = {
    17: {"donor_probe": "0.275 [0.210, 0.352]", "full": "0.195 [0.149, 0.250]", "lead": "0.843 [0.791, 0.884]"},
    23: {"donor_probe": "0.691 [0.613, 0.760]", "full": "0.263 [0.211, 0.322]", "lead": "0.894 [0.848, 0.927]"},
    28: {"donor_probe": "0.799 [0.727, 0.855]", "full": "0.174 [0.131, 0.227]", "lead": "0.877 [0.829, 0.913]"},
}
# Task-map reconstruction cosine in every OTHER pair — the band L23 should fall into if the
# L17 value was an artifact of a dead donor layer.
OTHER_PAIRS_TASKMAP_COS = {"gemma-2-9b -> gemma-2-2b": 0.197,
                           "Qwen2.5-7B -> gemma-2-2b": 0.253,
                           "instruction-tuned pair":   0.266}

print("L23 CHECK: donor =", MODEL_D, "| L_D =", L_D, "(was 17) | recipient =", MODEL_R,
      "| L_R =", L_R, "| donor dtype = bf16 | SMOKE_TEST =", SMOKE_TEST)

DONOR_ARM = llama | donor = meta-llama/Llama-3.1-8B L_D = 17 | recipient = google/gemma-2-2b L_R = 20 | SMOKE_TEST = False
L23 CHECK: donor = meta-llama/Llama-3.1-8B | L_D = 23 (was 17) | recipient = google/gemma-2-2b | L_R = 20 | donor dtype = bf16 | SMOKE_TEST = False


In [ ]:
# === CELL 3: Hugging Face login (Gemma and Llama are gated) ===
from huggingface_hub import login
login("")   # <-- paste your own read token here before running

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, full-answer) ===
# Copied VERBATIM from the base notebook. NOTE: `states_and_top` and `arith_fullanswer_correct`
# below close over a global `tokenizer`; CELL 6 aliases `tokenizer = tokenizer_r` so they remain
# correct if called, but the CROSS-FAMILY code in this notebook uses the explicitly
# tokenizer-parameterized replacements defined in CELL X0 instead.
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2

# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out

import re as _re
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None

# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok

print("helpers defined")

helpers defined


In [7]:
# === CELL X0: CROSS-FAMILY (dual-tokenizer) helpers — run right after CELL 5 ===
# Everything here exists because donor and recipient DO NOT share a tokenizer.
#
# DESIGN DECISION 1 — one problem, two encodings.
#   Each problem dict carries ids_d/tok_d (donor tokenization) and ids_r/tok_r (recipient
#   tokenization) of the SAME text. `ids_*` ends at that model's own LAST PROMPT TOKEN, i.e. the
#   position whose logits predict the first answer token. On Gemma that is the standalone leading
#   space; on Qwen/Llama the space fuses with the first digit so it is the "=" token. Different
#   surface positions, identical operative role. Only this ONE position is grafted, so the two
#   token sequences never need to align.
#
# DESIGN DECISION 2 — scoring is tokenizer-agnostic.
#   No token ID is compared across models anywhere. gen_score_arith greedily generates from the
#   recipient and scores the DECODED text: full-answer correctness and leading-digit correctness.
#   The old "first-token conferral" metric is NOT valid cross-family and is not reported as a
#   headline. `tok_r` is still used, but only as the CE target when training the map — that is a
#   purely within-recipient quantity in the recipient's own vocabulary, which is legitimate.
import torch, torch.nn.functional as F, numpy as np, random, json

def fd(x):
    """First (leading) digit of an integer answer, 0-9. Defined here because the notebooks this
    is assembled from define it only in cells we do not include."""
    return int(str(abs(int(round(x))))[0]) if x is not None else 0

def probe_first_digit(Xtr, ytr, Xte, yte, steps=300, nclass=10, lr=1e-2):
    """Linear probe on a hidden state -> class label. Returns PREDICTIONS on Xte."""
    Pw = torch.zeros(Xtr.shape[1], nclass, requires_grad=True)
    opt = torch.optim.Adam([Pw], lr=lr); mu = Xtr.mean(0); A, Bx = Xtr-mu, Xte-mu
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy(A @ Pw, ytr).backward(); opt.step()
    return (Bx @ Pw.detach()).argmax(1)

def probe_split_acc(X, y, nclass=10):
    """Half/half split probe accuracy as a Wilson-CI string. X, y are CPU tensors."""
    if len(y) < 40: return "n/a (n<40)"
    h = len(y)//2
    pred = probe_first_digit(X[:h].float(), y[:h], X[h:].float(), y[h:], nclass=nclass)
    return fmt(wilson_bools((pred == y[h:]).tolist()))

# ---- tokenizer-parameterized state collection (replaces CELL 5's states_and_top) ----
@torch.inference_mode()
def states_and_top_tok(model, tok, layers, prob_ids, batch=None):
    """Left-pad with THIS model's own pad id, forward once, take the last-position residual at
    each requested layer. Returns ({layer: [N,d]}, top_token_ids). The top ids are a diagnostic
    only; they are never compared across models."""
    batch = batch or ARITH_BATCH
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tok.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- arithmetic data with BOTH encodings ----
def gen_arith_dual(n, rng, exclude=None):
    """Generate a*b+c problems that BOTH tokenizers can encode, so donor and recipient see an
    identical problem set. Drops a problem if either tokenizer is not prefix-consistent."""
    exclude = exclude or set(); out, seen = [], set(); tries = 0; dropped = 0
    while len(out) < n and tries < n*200:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids_d, tid_d = _aencode(tokenizer_d, expr, ans)
        ids_r, tid_r = _aencode(tokenizer_r, expr, ans)
        if tid_d is None or tid_r is None:
            dropped += 1; continue
        out.append(dict(expr=expr, ans=ans, ids_d=ids_d, tok_d=tid_d, ids_r=ids_r, tok_r=tid_r))
    if dropped: print(f"  gen_arith_dual: dropped {dropped} problems (tokenizer prefix-inconsistent)")
    return out

# ---- THE scoring function: decoded-answer metrics only ----
@torch.inference_mode()
def gen_score_arith(model, tok, layer, probs, ids_key, vecs=None, batch=None, max_new=None):
    """Greedy-generate from `model` using its OWN encoding (probs[i][ids_key]) and score the
    DECODED text. If vecs is given, vecs[i] is grafted at the last prompt position during prefill.
    Returns {"full": [bool], "lead": [bool]} -- (a) whole-answer string correctness,
    (b) leading-digit correctness. Tokenizer-agnostic by construction."""
    batch = batch or ARITH_BATCH; max_new = max_new or MAX_NEW_ARITH
    full, lead, handle = [], [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p[ids_key] for p in chunk], tok.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=max_new,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            txt = tok.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                full.append(pred is not None and abs(pred - p["ans"]) < 0.5)
                lead.append(pred is not None and fd(pred) == fd(p["ans"]))
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return {"full": full, "lead": lead}

def score_pair(sc):
    """Format a {'full':..,'lead':..} score dict as two Wilson-CI strings."""
    return {"full": fmt(wilson_bools(sc["full"])), "lead": fmt(wilson_bools(sc["lead"]))}

print("cross-family helpers defined (fd, probe_first_digit, probe_split_acc, "
      "states_and_top_tok, gen_arith_dual, gen_score_arith, score_pair)")

cross-family helpers defined (fd, probe_first_digit, probe_split_acc, states_and_top_tok, gen_arith_dual, gen_score_arith, score_pair)


In [8]:
# === CELL 6: load BOTH models and BOTH tokenizers; donor in bf16 (NOT 4-bit) ===
# CHANGE vs every earlier notebook: two separate tokenizers, and the same-tokenizer assert that
# used to live at the bottom of this cell is DELETED — it would fire on every cross-family pair
# and is not applicable, because we graft exactly one position that each model locates for itself.
tokenizer_r = AutoTokenizer.from_pretrained(MODEL_R)     # RECIPIENT tokenizer (gemma-2-2b)
tokenizer_d = AutoTokenizer.from_pretrained(MODEL_D)     # DONOR tokenizer (Qwen / Llama)
for _t in (tokenizer_r, tokenizer_d):
    _t.padding_side = "left"
    if _t.pad_token is None: _t.pad_token = _t.eos_token
# Back-compat alias so any verbatim-copied CELL 5 helper that references a bare `tokenizer`
# operates on the RECIPIENT (all generation in this notebook happens on the recipient).
tokenizer = tokenizer_r

# QUANTIZE_DONOR: keep False. Under 4-bit this transformers/bnb build runs the unquantized layers
# in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) -> NaN logits -> argmax collapses to
# token 0. bf16 has the exponent range to avoid this. A 7-8B donor is ~15-16 GB in bf16, which
# fits alongside the 5 GB recipient on a 48 GB card with room to spare.
QUANTIZE_DONOR = globals().get("QUANTIZE_DONOR", False)

model_r = AutoModelForCausalLM.from_pretrained(
    MODEL_R, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_DONOR:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_d = AutoModelForCausalLM.from_pretrained(
        MODEL_D, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_d = AutoModelForCausalLM.from_pretrained(
        MODEL_D, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()

N_LAYERS_D = model_d.config.num_hidden_layers
N_LAYERS_R = model_r.config.num_hidden_layers
D_DIM = model_d.config.hidden_size
R_DIM = model_r.config.hidden_size
print(f"loaded recipient {MODEL_R}: {N_LAYERS_R} layers, d={R_DIM}")
print(f"loaded donor     {MODEL_D}: {N_LAYERS_D} layers, d={D_DIM} | "
      f"{'4-bit' if QUANTIZE_DONOR else 'bf16'}")
assert 0 <= L_R < N_LAYERS_R, f"L_R={L_R} out of range for recipient"
assert 0 <= L_D < N_LAYERS_D, f"L_D={L_D} out of range for donor"
DONOR_SWEEP_LAYERS = sorted({L for L in DONOR_SWEEP_LAYERS if 0 <= L < N_LAYERS_D} | {L_D})
print("donor sweep layers:", DONOR_SWEEP_LAYERS)

# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# hundreds of silently-filtered problems later. A healthy donor tops a real word here.
with torch.inference_mode():
    _hl = model_d(tokenizer_d("The capital of France is",
                              return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "donor produced NaN/Inf logits — numerical blowup. If QUANTIZE_DONOR=True, the donor is "
    "overflowing fp16; set QUANTIZE_DONOR=False to load it in bf16 (needs the VRAM but is safe).")
print("donor health check ok; donor top token:", repr(tokenizer_d.decode([_hl.argmax().item()])))
del _hl

# Show the tokenizer divergence explicitly — this is the whole reason for CELL X0.
_probe = "3 * 12 + 7 = 43"
print("recipient ids:", tokenizer_r(_probe).input_ids)
print("donor     ids:", tokenizer_d(_probe).input_ids)
print("NOTE: sequences differ. That is expected and fine — only ONE position is grafted, and each "
      "model locates that position (its own last prompt token) independently.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

loaded recipient google/gemma-2-2b: 26 layers, d=2304
loaded donor     meta-llama/Llama-3.1-8B: 32 layers, d=4096 | bf16
donor sweep layers: [23]
donor health check ok; donor top token: ' a'
recipient ids: [2, 235304, 649, 235248, 235274, 235284, 963, 235248, 235324, 589, 235248, 235310, 235304]
donor     ids: [128000, 18, 353, 220, 717, 489, 220, 22, 284, 220, 3391]
NOTE: sequences differ. That is expected and fine — only ONE position is grafted, and each model locates that position (its own last prompt token) independently.


In [9]:
# === CELL X1: arithmetic data, DONOR-SOLVED and RECIPIENT-UNSOLVABLE bins, reconstruction map ===
# Both filters are decoded-answer checks on each model's OWN tokenization. No token IDs cross the
# family boundary. The unsolvable bin = {donor gets it right} AND {recipient gets it wrong}, so any
# success after the graft was conferred by the stitch.
train = gen_arith_dual(N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith_dual(N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")

# ---- donor states + DONOR-SOLVED filter (donor's own encoding, donor's own decoded answer) ----
Xd_t_all, _ = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in train])
X_dt = Xd_t_all[L_D]
Xd_e_all, _ = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in evalp])
X_de_raw = Xd_e_all[L_D]
donor_sc = gen_score_arith(model_d, tokenizer_d, L_D, evalp, "ids_d", vecs=None)
keep = [i for i in range(len(evalp)) if donor_sc["full"][i]]
print(f"  donor solves {len(keep)}/{len(evalp)} "
      f"(full {fmt(wilson_bools(donor_sc['full']))}, lead {fmt(wilson_bools(donor_sc['lead']))})")
evalp = [evalp[i] for i in keep]; X_de = X_de_raw[keep]
del Xd_e_all, X_de_raw

# ---- recipient states + RECIPIENT-UNSOLVABLE filter (recipient's own encoding + decoded answer) ----
Xr_t_all, _ = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in train])
X_rt = Xr_t_all[L_R]
Xr_e_all, r_top = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in evalp])
X_re = Xr_e_all[L_R]
native_sc = gen_score_arith(model_r, tokenizer_r, L_R, evalp, "ids_r", vecs=None)
solvable = native_sc["full"]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv   = [i for i in range(len(evalp)) if solvable[i]]
print(f"  recipient natively solves {len(solv)}/{len(evalp)} -> UNSOLVABLE BIN n={len(unsolv)}")
del Xd_t_all, Xr_t_all, Xr_e_all

# ---- reconstruction map (ridge, donor L_D -> recipient L_R) ----
mu_d, mu_r, Wr = fit_ridge(X_dt, X_rt)
mu_dd, mu_rd = mu_d.to(DEVICE), mu_r.to(DEVICE)
recon_map = (mu_dd, mu_rd, Wr.to(DEVICE))
def map_recon(xd):
    m9, m2, W = recon_map; return (xd.to(DEVICE)-m9) @ W + m2

# ---- the one call every experiment uses to score a graft ----
def confer(vecs, idxs, probs=None):
    """Graft vecs[k] into problem probs[idxs[k]] on the recipient and score DECODED output.
    `vecs` is a [len(idxs), R_DIM] tensor. Returns {'full': [bool], 'lead': [bool]}."""
    probs = evalp if probs is None else probs
    return gen_score_arith(model_r, tokenizer_r, L_R, [probs[j] for j in idxs], "ids_r",
                           vecs=list(vecs.detach().float().cpu()))

# ---- donor leading-digit probe at L_D (always computed; the MLP/ceiling cells reference it) ----
_y_all = torch.tensor([fd(p["ans"]) for p in evalp])
donor_probe_LD = probe_split_acc(X_de, _y_all)
RESULTS["setup"] = {
    "donor": MODEL_D, "recipient": MODEL_R, "L_D": L_D, "L_R": L_R,
    "d_donor": int(X_dt.shape[1]), "d_recipient": int(X_rt.shape[1]),
    "n_train": len(train), "n_eval_donor_solved": len(evalp),
    "n_unsolvable": len(unsolv), "n_solvable": len(solv),
    "donor_full_on_all_eval": fmt(wilson_bools(donor_sc["full"])),
    "recipient_native_full_donor_solved": fmt(wilson_bools(native_sc["full"])),
    "recipient_native_lead_donor_solved": fmt(wilson_bools(native_sc["lead"])),
    "recipient_native_lead_on_unsolv": fmt(wilson_bools([native_sc["lead"][i] for i in unsolv])),
    f"donor_leading_digit_probe_L{L_D}": donor_probe_LD,
    "_note": "all bins/metrics are decoded-answer based; no token id is compared across families",
}
print(json.dumps(RESULTS["setup"], indent=2))

arith: 3000 train, 2000 eval


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  donor solves 297/2000 (full 0.148 [0.134, 0.165], lead 0.726 [0.706, 0.745])
  recipient natively solves 61/297 -> UNSOLVABLE BIN n=236
{
  "donor": "meta-llama/Llama-3.1-8B",
  "recipient": "google/gemma-2-2b",
  "L_D": 23,
  "L_R": 20,
  "d_donor": 4096,
  "d_recipient": 2304,
  "n_train": 3000,
  "n_eval_donor_solved": 297,
  "n_unsolvable": 236,
  "n_solvable": 61,
  "donor_full_on_all_eval": "0.148 [0.134, 0.165]",
  "recipient_native_full_donor_solved": "0.205 [0.163, 0.255]",
  "recipient_native_lead_donor_solved": "0.566 [0.509, 0.621]",
  "recipient_native_lead_on_unsolv": "0.453 [0.391, 0.517]",
  "donor_leading_digit_probe_L23": "0.691 [0.613, 0.760]",
  "_note": "all bins/metrics are decoded-answer based; no token id is compared across families"
}


In [10]:
# === CELL X3: train the task-supervised map at L_D, 5 seeds (the only stochastic part) ===
# Warm-started at the ridge reconstruction map. Trained by CE on the RECIPIENT's own first answer
# token (within-recipient vocabulary -> valid cross-family). Scoring later is decoded-text only.
task_maps = []
model_r.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu_r.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                xd = X_dt[sub].to(DEVICE)
                _graft["vec"] = (xd - mu_dd) @ W + b
                ids, m = left_pad([train[k]["ids_r"] for k in sub], tokenizer_r.pad_token_id)
                lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok_r"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}")
model_r.requires_grad_(True)

def map_task(i):
    W, b = task_maps[i]
    return lambda xd: (xd.to(DEVICE)-mu_dd) @ W + b
print(f"trained {len(task_maps)} task maps at donor L{L_D} -> recipient L{L_R}")

  seed 0: final batch CE 0.005
  seed 1: final batch CE 0.051
  seed 2: final batch CE 0.018
  seed 3: final batch CE 0.007
  seed 4: final batch CE 0.004
trained 5 task maps at donor L23 -> recipient L20


In [11]:
# === CELL X4: EXP 2 — CORE CHANNEL: recon / task (5 seeds) / SHUFFLE / self-graft ===
# SHUFFLE is the critical control here. A cross-family map has far more room to learn a generic
# task prior than a within-family one, so if task ~= shuffle, that is the clean NEGATIVE result:
# the map learned "emit a plausible product", not "read this donor's answer".
# Self-graft = re-inject the recipient's OWN native L_R state. It must reproduce native behaviour
# (~0 on the unsolvable bin, by the bin's definition); it is the hook-machinery sanity check and
# proves the graft site itself confers nothing.
# Based on CELLs 8/9/10 of consolidated_eval.ipynb, rewritten for decoded-answer scoring.
if RUN_CORE and unsolv:
    arr = {"n_unsolv": len(unsolv), "n_solv": len(solv)}

    # continuous diagnostic: how close does each map land to the recipient's real state?
    rc_recon = F.cosine_similarity(map_recon(X_de).cpu(), X_re, dim=1).numpy()
    rc_task  = F.cosine_similarity(map_task(0)(X_de).detach().cpu(), X_re, dim=1).numpy()
    arr["recon_cos_reconmap"] = fmt(bootstrap_ci(rc_recon))
    arr["recon_cos_taskmap"]  = fmt(bootstrap_ci(rc_task))

    # ---- headline bin: UNSOLVABLE ----
    arr["native_unsolv"]  = {"full": "0.000 [by construction]",
                             "lead": fmt(wilson_bools([native_sc["lead"][i] for i in unsolv]))}
    arr["selfgraft_unsolv"] = score_pair(confer(X_re[unsolv].to(DEVICE), unsolv))
    arr["recon_unsolv"]     = score_pair(confer(map_recon(X_de[unsolv]), unsolv))

    seed_full, seed_lead = [], []
    for s in range(len(task_maps)):
        sc = confer(map_task(s)(X_de[unsolv]), unsolv)
        seed_full.append(float(np.mean(sc["full"]))); seed_lead.append(float(np.mean(sc["lead"])))
        if s == 0: arr["task_unsolv_seed0"] = score_pair(sc)
    arr["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
    arr["task_unsolv_lead_acrossseed"] = fmt(across_seed_ci(seed_lead))

    # SHUFFLE: task map fed the WRONG problem's donor state (content-specificity control)
    g = torch.Generator().manual_seed(0)
    shuf = torch.randperm(len(unsolv), generator=g)
    shuf_idx = [unsolv[int(k)] for k in shuf]
    arr["shuffle_unsolv"] = score_pair(confer(map_task(0)(X_de[shuf_idx]), unsolv))

    # ---- sanity bin: SOLVABLE (the graft must not DESTROY what the recipient already knows) ----
    if solv:
        arr["recon_solv"] = score_pair(confer(map_recon(X_de[solv]), solv))
        arr["task_solv"]  = score_pair(confer(map_task(0)(X_de[solv]), solv))
    arr["_read"] = ("task >> shuffle => the stitch transfers donor-specific content. "
                    "task ~= shuffle => the cross-family map only learned a generic task prior.")
    RESULTS["arithmetic_core"] = arr
    print("EXP2 core channel:", json.dumps(arr, indent=2))
else:
    print("EXP2 skipped (RUN_CORE=False or empty unsolvable bin).")

EXP2 core channel: {
  "n_unsolv": 236,
  "n_solv": 61,
  "recon_cos_reconmap": "0.925 [0.920, 0.930]",
  "recon_cos_taskmap": "0.808 [0.801, 0.815]",
  "native_unsolv": {
    "full": "0.000 [by construction]",
    "lead": "0.453 [0.391, 0.517]"
  },
  "selfgraft_unsolv": {
    "full": "0.000 [0.000, 0.016]",
    "lead": "0.449 [0.387, 0.513]"
  },
  "recon_unsolv": {
    "full": "0.097 [0.066, 0.142]",
    "lead": "0.572 [0.508, 0.634]"
  },
  "task_unsolv_seed0": {
    "full": "0.263 [0.211, 0.322]",
    "lead": "0.898 [0.853, 0.931]"
  },
  "task_unsolv_full_acrossseed": "0.241 [0.224, 0.257]",
  "task_unsolv_lead_acrossseed": "0.886 [0.871, 0.900]",
  "shuffle_unsolv": {
    "full": "0.004 [0.001, 0.024]",
    "lead": "0.119 [0.083, 0.166]"
  },
  "recon_solv": {
    "full": "0.590 [0.465, 0.705]",
    "lead": "0.721 [0.598, 0.818]"
  },
  "task_solv": {
    "full": "0.672 [0.547, 0.777]",
    "lead": "0.902 [0.802, 0.954]"
  },
  "_read": "task >> shuffle => the stitch transfers d

In [12]:
# === CELL X4b: extra cosines (per seed, and restricted to the unsolvable bin) ===
# CELL X4 computes the reconstruction cosine over ALL donor-solved eval problems with the seed-0
# task map -- that is exactly how the L17 value of 0.824 was produced, so it is kept untouched for
# comparability. These extras answer two follow-ups cheaply: is the cosine stable across the 5
# seeds, and does it look different on the unsolvable bin specifically (which is where every
# conferral number is measured)? No model forwards; both states are already on CPU.
if RUN_CORE and unsolv and "arithmetic_core" in RESULTS:
    _core = RESULTS["arithmetic_core"]
    _core["taskmap_cos_by_seed"] = [
        round(float(F.cosine_similarity(map_task(s)(X_de).detach().cpu(), X_re, dim=1).mean()), 4)
        for s in range(len(task_maps))
    ]
    _cu = F.cosine_similarity(map_task(0)(X_de[unsolv]).detach().cpu(), X_re[unsolv], dim=1).numpy()
    _ru = F.cosine_similarity(map_recon(X_de[unsolv]).cpu(), X_re[unsolv], dim=1).numpy()
    _core["recon_cos_taskmap_unsolv"]  = fmt(bootstrap_ci(_cu))
    _core["recon_cos_reconmap_unsolv"] = fmt(bootstrap_ci(_ru))
    _core["_cos_note"] = ("recon_cos_* (unsuffixed) are over ALL donor-solved eval with the seed-0 "
                          "task map -- identical recipe to the L17 run's 0.824, so directly "
                          "comparable. *_unsolv restricts to the unsolvable bin.")
    print("task-map cosine by seed:", _core["taskmap_cos_by_seed"])
    print("unsolvable bin only -> taskmap", _core["recon_cos_taskmap_unsolv"],
          "| reconmap", _core["recon_cos_reconmap_unsolv"])
else:
    print("X4b skipped (core eval did not run).")

task-map cosine by seed: [0.8076, 0.804, 0.8044, 0.8062, 0.8018]
unsolvable bin only -> taskmap 0.819 [0.812, 0.826] | reconmap 0.933 [0.928, 0.937]


In [13]:
# === CELL X7: EXP 5 — TRANSCRIPTION PROBE (leading digit on donor state AND on the stitched vector) ===
# The graft REPLACES the recipient's L_R state with the map output, so "the recipient's state after
# the graft" at that layer IS the map output. Probe the answer's LEADING DIGIT from:
#   native recipient state (no graft) -- should NOT carry the answer on the unsolvable bin,
#   reconstruction-map output, task-map output (the stitched vector), and the donor state (ceiling).
# A large native->task gap = the map WROTE the answer into the recipient's stream (transcription),
# rather than the recipient computing it. Leading digit is used because it is the tokenizer-agnostic
# unit: it is a single token in the recipient's vocabulary regardless of the donor's family.
# Cheap: no model forwards, just linear probes on states already in memory.
# Based on CELL 14 of consolidated_eval.ipynb.
if RUN_TRANSCRIBE:
    pidx = unsolv if len(unsolv) >= 60 else list(range(len(evalp)))
    y_tc = torch.tensor([fd(evalp[i]["ans"]) for i in pidx])
    Xr_sub, Xd_sub = X_re[pidx], X_de[pidx]
    task_out  = map_task(0)(Xd_sub).detach().cpu()
    recon_out = map_recon(Xd_sub).detach().cpu()
    RESULTS["transcription_probe"] = {
        "n": len(pidx), "bin": "unsolvable" if pidx is unsolv else "all_donor_solved",
        f"native_recipient_L{L_R}": probe_split_acc(Xr_sub, y_tc),
        "recon_map_output":         probe_split_acc(recon_out, y_tc),
        "task_map_output_STITCHED": probe_split_acc(task_out, y_tc),
        f"donor_L{L_D}_reference":  probe_split_acc(Xd_sub, y_tc),
        "_read": "native low + stitched ~= donor => the map transcribed the donor's answer; "
                 "stitched ~= native => nothing about the answer crossed the family boundary.",
    }
    print("EXP5 transcription probe:", json.dumps(RESULTS["transcription_probe"], indent=2))
else:
    print("EXP5 skipped (RUN_TRANSCRIBE=False).")

EXP5 transcription probe: {
  "n": 236,
  "bin": "unsolvable",
  "native_recipient_L20": "0.449 [0.362, 0.539]",
  "recon_map_output": "0.636 [0.546, 0.717]",
  "task_map_output_STITCHED": "0.695 [0.607, 0.771]",
  "donor_L23_reference": "0.720 [0.633, 0.793]",
  "_read": "native low + stitched ~= donor => the map transcribed the donor's answer; stitched ~= native => nothing about the answer crossed the family boundary."
}


In [14]:
# === CELL CMP: L17 vs L23 SIDE-BY-SIDE — does each anomaly persist or resolve? ===
# Nothing here touches a model. It reads RESULTS (filled by X1/X4/X7 above) and prints it against
# L17_REF (the completed L_D=17 run, hardcoded in CELL 2). Point estimates are parsed out of the
# "p [lo, hi]" strings for the deltas; the full CI strings are printed as-is.
import re as _re_cmp

def _pt(s):
    """Point estimate out of a 'p [lo, hi]' string (or a bare number). NaN if absent."""
    if s is None: return float("nan")
    if isinstance(s, (int, float)): return float(s)
    m = _re_cmp.search(r"-?\d+\.\d+", str(s))
    return float(m.group()) if m else float("nan")

def _d(new, old):
    a, b = _pt(new), _pt(old)
    if a != a or b != b: return "   n/a "
    return f"{a-b:+7.3f}"

def _row(label, old, new):
    print(f"  {label:<34} {str(old):<24} {str(new):<24} {_d(new, old)}")

def _hdr(title):
    print()
    print("-"*104)
    print(f"  {title}")
    print("-"*104)
    print(f"  {'quantity':<34} {'L_D = 17 (completed)':<24} {'L_D = 23 (this run)':<24} {'delta':>7}")

core = RESULTS.get("arithmetic_core", {})
tp   = RESULTS.get("transcription_probe", {})
setup = RESULTS.get("setup", {})

print("="*104)
print("  CROSS-FAMILY L17 vs L23   |   donor meta-llama/Llama-3.1-8B  ->  recipient google/gemma-2-2b")
print(f"  recipient layer L_R = {L_R} in BOTH runs; donor layer is the only thing that changed")
print("="*104)
print(f"  bins   L17: n_donor_solved={L17_REF['n_eval_donor_solved']}, n_unsolvable={L17_REF['n_unsolv']}"
      f"   |   L23: n_donor_solved={setup.get('n_eval_donor_solved','?')}, "
      f"n_unsolvable={setup.get('n_unsolv', core.get('n_unsolv','?'))}")
print("  (the bins are set by ungrafted generation, so they should match; if they do not, the "
      "comparison is still valid\n   but no longer problem-for-problem identical)")

# ---------------- ANOMALY 1 ----------------
_hdr("ANOMALY 1 — reconstruction cosine to the recipient's natural manifold  (all donor-solved eval)")
_row("recon_cos_reconmap", L17_REF["recon_cos_reconmap"], core.get("recon_cos_reconmap"))
_row("recon_cos_taskmap  <<< HEADLINE", L17_REF["recon_cos_taskmap"], core.get("recon_cos_taskmap"))
print()
print("  reference band, task map in every OTHER pair:")
for k, v in OTHER_PAIRS_TASKMAP_COS.items():
    print(f"    {k:<32} {v:.3f}")
if "taskmap_cos_by_seed" in core:
    print("  this run, task-map cosine by seed: " +
          ", ".join(f"s{i}={c:.3f}" for i, c in enumerate(core["taskmap_cos_by_seed"])))
if "recon_cos_taskmap_unsolv" in core:
    print(f"  this run, restricted to the unsolvable bin: "
          f"taskmap={core['recon_cos_taskmap_unsolv']}  reconmap={core['recon_cos_reconmap_unsolv']}")

_new_cos = _pt(core.get("recon_cos_taskmap"))
_band_hi = max(OTHER_PAIRS_TASKMAP_COS.values())
print()
if _new_cos != _new_cos:
    v1 = "INCONCLUSIVE — core eval did not run."
elif _new_cos <= _band_hi + 0.10:
    v1 = (f"RESOLVED. At L23 the task map's cosine is {_new_cos:.3f}, inside/next to the "
          f"0.20-0.27 band of every other pair.\n           The L17 value of 0.824 was an artifact "
          f"of a donor layer with almost no linearly-readable answer:\n           the map never "
          f"left its ridge warm start. Conferral does coincide with leaving the manifold.")
elif _new_cos >= 0.60:
    v1 = (f"PERSISTS. At L23 the task map's cosine is still {_new_cos:.3f}, far above the "
          f"0.20-0.27 band.\n           This is a real property of the Llama->Gemma cross-family "
          f"channel, not a bad-layer artifact:\n           high conferral WITHOUT leaving the "
          f"recipient's manifold. The paper's claim needs qualifying.")
else:
    v1 = (f"PARTIAL. Cosine moved to {_new_cos:.3f} — between the other-pair band ({_band_hi:.3f}) "
          f"and the L17 value (0.824).\n           The donor layer explains some but not all of the "
          f"anomaly. Report the gradient, not a verdict.")
print(f"  VERDICT 1: {v1}")

# ---------------- ANOMALY 2 ----------------
_hdr(f"ANOMALY 2 — transcription probe, leading digit, UNSOLVABLE bin (n={tp.get('n','?')})")
_row(f"recipient native at L_R={L_R}", L17_REF["probe_native_L20"], tp.get(f"native_recipient_L{L_R}"))
_row("recon-map output", L17_REF["probe_recon_map"], tp.get("recon_map_output"))
_row("task-map STITCHED  <<< HEADLINE", L17_REF["probe_stitched"], tp.get("task_map_output_STITCHED"))
_row("donor state at L_D", L17_REF["probe_donor_LD"], tp.get(f"donor_L{L_D}_reference"))
print()
print(f"  donor's own leading-digit probe at its graft layer (from CELL X1 setup):")
print(f"    L17: {L17_REF['donor_probe_at_LD']}   ->   L23: "
      f"{setup.get(f'donor_leading_digit_probe_L{L_D}', 'n/a')}")
print(f"    (the L17 run's sweep predicted L23 = {L17_SWEEP[23]['donor_probe']})")

_st, _na = _pt(tp.get("task_map_output_STITCHED")), _pt(tp.get(f"native_recipient_L{L_R}"))
print()
if _st != _st or _na != _na:
    v2 = "INCONCLUSIVE — transcription probe did not run."
elif _st > _na + 0.02:
    v2 = (f"RESOLVED. Stitched {_st:.3f} > native {_na:.3f}. The stitched vector now reads the "
          f"answer BETTER than the\n           recipient's own state, as in every other pair. The "
          f"L17 inversion was a dead-donor-layer artifact.")
elif _st < _na - 0.02:
    v2 = (f"PERSISTS. Stitched {_st:.3f} < native {_na:.3f}, still inverted relative to every other "
          f"pair.\n           Whatever crosses this family boundary is NOT linearly-readable answer "
          f"content, even when the\n           donor state itself carries it. Conferral here is not "
          f"transcription.")
else:
    v2 = (f"PARTIAL. Stitched {_st:.3f} ~= native {_na:.3f} — the inversion is gone but the stitched "
          f"vector does not\n           beat native either. Intermediate; report as such.")
print(f"  VERDICT 2: {v2}")

# ---------------- conferral ----------------
_hdr("CONFERRAL on the unsolvable bin — LEADING DIGIT")
_row("native (no graft)", L17_REF["native_unsolv_lead"], (core.get("native_unsolv") or {}).get("lead"))
_row("self-graft control", L17_REF["selfgraft_lead"], (core.get("selfgraft_unsolv") or {}).get("lead"))
_row("recon map", L17_REF["recon_lead"], (core.get("recon_unsolv") or {}).get("lead"))
_row("task map, seed 0", L17_REF["task_seed0_lead"], (core.get("task_unsolv_seed0") or {}).get("lead"))
_row("task map, 5 seeds pooled", L17_REF["task_lead_acrossseed"], core.get("task_unsolv_lead_acrossseed"))
_row("shuffle control", L17_REF["shuffle_lead"], (core.get("shuffle_unsolv") or {}).get("lead"))

_hdr("CONFERRAL on the unsolvable bin — FULL ANSWER")
_row("native (no graft)", "0.000 [by construction]", (core.get("native_unsolv") or {}).get("full"))
_row("self-graft control", L17_REF["selfgraft_full"], (core.get("selfgraft_unsolv") or {}).get("full"))
_row("recon map", L17_REF["recon_full"], (core.get("recon_unsolv") or {}).get("full"))
_row("task map, seed 0", L17_REF["task_seed0_full"], (core.get("task_unsolv_seed0") or {}).get("full"))
_row("task map, 5 seeds pooled", L17_REF["task_full_acrossseed"], core.get("task_unsolv_full_acrossseed"))
_row("shuffle control", L17_REF["shuffle_full"], (core.get("shuffle_unsolv") or {}).get("full"))
print()
print(f"  the L17 run's sweep predicted L23 seed-0 conferral: full {L17_SWEEP[23]['full']}, "
      f"lead {L17_SWEEP[23]['lead']}")
print("  (that sweep trained its own seed-0 map at L23 with the same recipe, so it is a genuine")
print("   out-of-sample prediction for the seed-0 numbers above — a large miss means something")
print("   other than the donor layer changed.)")

# ---------------- sanity ----------------
_hdr("SANITY — the graft must not destroy what the recipient already knows (solvable bin)")
_row("recon map, full", "0.361 [0.252, 0.486]", (core.get("recon_solv") or {}).get("full"))
_row("task map, full",  "0.574 [0.449, 0.690]", (core.get("task_solv") or {}).get("full"))

RESULTS["l17_vs_l23"] = {
    "L17_reference": L17_REF,
    "L17_donor_layer_sweep": L17_SWEEP,
    "other_pairs_taskmap_cos": OTHER_PAIRS_TASKMAP_COS,
    "anomaly_1_manifold": {
        "L17_recon_cos_taskmap": L17_REF["recon_cos_taskmap"],
        "L23_recon_cos_taskmap": core.get("recon_cos_taskmap"),
        "verdict": v1.split("\n")[0],
    },
    "anomaly_2_transcription": {
        "L17_stitched": L17_REF["probe_stitched"], "L17_native": L17_REF["probe_native_L20"],
        "L23_stitched": tp.get("task_map_output_STITCHED"),
        "L23_native": tp.get(f"native_recipient_L{L_R}"),
        "verdict": v2.split("\n")[0],
    },
}
print()
print("="*104)
print("  comparison stored in RESULTS['l17_vs_l23'] — the SAVE cell writes it out")
print("="*104)

  CROSS-FAMILY L17 vs L23   |   donor meta-llama/Llama-3.1-8B  ->  recipient google/gemma-2-2b
  recipient layer L_R = 20 in BOTH runs; donor layer is the only thing that changed
  bins   L17: n_donor_solved=297, n_unsolvable=236   |   L23: n_donor_solved=297, n_unsolvable=236
  (the bins are set by ungrafted generation, so they should match; if they do not, the comparison is still valid
   but no longer problem-for-problem identical)

--------------------------------------------------------------------------------------------------------
  ANOMALY 1 — reconstruction cosine to the recipient's natural manifold  (all donor-solved eval)
--------------------------------------------------------------------------------------------------------
  quantity                           L_D = 17 (completed)     L_D = 23 (this run)        delta
  recon_cos_reconmap                 0.906 [0.900, 0.912]     0.925 [0.920, 0.930]      +0.019
  recon_cos_taskmap  <<< HEADLINE    0.824 [0.816, 0.832]     0

In [15]:
# === CELL SAVE: concentrate everything and write crossfamily_llama_L23_check.json ===
keys = [k for k in ("setup", "arithmetic_core", "transcription_probe", "l17_vs_l23")
        if k in RESULTS]
out = {k: RESULTS[k] for k in keys}
out["_config"] = {"donor_arm": DONOR_ARM, "donor": MODEL_D, "recipient": MODEL_R,
                  "L_D": L_D, "L_R": L_R, "smoke_test": SMOKE_TEST,
                  "task_seeds": TASK_SEEDS, "task_epochs": TASK_EPOCHS,
                  "purpose": "re-run of the completed L_D=17 cross-family pair at L_D=23 to test "
                             "whether the on-manifold task map (recon_cos_taskmap=0.824) and the "
                             "below-native stitched probe (0.254 vs 0.449) are real or artifacts "
                             "of a donor layer where the answer is not linearly available",
                  "compare_against": "crossfamily_results_Llama-3.1-8B.json (the L_D=17 run)",
                  "scoring": "decoded-answer only (full-answer string + leading digit); "
                             "no token id compared across families"}
fname = "crossfamily_llama_L23_check.json"
with open(fname, "w") as f: json.dump(out, f, indent=2)
print("="*72); print("ALL RESULTS (point [95% Wilson CI])"); print("="*72)
print(json.dumps(out, indent=2))
print("saved", fname, "->  DOWNLOAD before terminating the pod")

ALL RESULTS (point [95% Wilson CI])
{
  "setup": {
    "donor": "meta-llama/Llama-3.1-8B",
    "recipient": "google/gemma-2-2b",
    "L_D": 23,
    "L_R": 20,
    "d_donor": 4096,
    "d_recipient": 2304,
    "n_train": 3000,
    "n_eval_donor_solved": 297,
    "n_unsolvable": 236,
    "n_solvable": 61,
    "donor_full_on_all_eval": "0.148 [0.134, 0.165]",
    "recipient_native_full_donor_solved": "0.205 [0.163, 0.255]",
    "recipient_native_lead_donor_solved": "0.566 [0.509, 0.621]",
    "recipient_native_lead_on_unsolv": "0.453 [0.391, 0.517]",
    "donor_leading_digit_probe_L23": "0.691 [0.613, 0.760]",
    "_note": "all bins/metrics are decoded-answer based; no token id is compared across families"
  },
  "arithmetic_core": {
    "n_unsolv": 236,
    "n_solv": 61,
    "recon_cos_reconmap": "0.925 [0.920, 0.930]",
    "recon_cos_taskmap": "0.808 [0.801, 0.815]",
    "native_unsolv": {
      "full": "0.000 [by construction]",
      "lead": "0.453 [0.391, 0.517]"
    },
    "selfgraf